## PROJECT - AIRLINE AI ASSITANT

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from rich import print as rich_print

In [2]:
# Load environment variables from .env file
load_dotenv(override=True)

# Set up Groq API client
groq_api_key = os.getenv("GROQ_API_KEY")
groq_model = os.getenv("GROQ_MODEL") or "openai/gpt-oss-20b"
groq_base_url = os.getenv("GROQ_BASE_URL")

# Check if the API key and base URL are set
if groq_api_key is None:
    raise ValueError("GROQ_API_KEY environment variable is not set.")
else: 
    rich_print(f"[green]GROQ_API_KEY loaded successfully[/green]: [yellow]{groq_api_key[:8]}...[/yellow]")


openai = OpenAI(base_url=groq_base_url, api_key=groq_api_key)

GROQ_API_KEY loaded successfully: gsk_GIUx...

In [3]:
system_message = """
You are a helpful assistant for an Airline called Akasha Air.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]}for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    rich_print(f"[green]Constructed messages for API:[/green] [yellow]{messages}[/yellow]")
    response = openai.chat.completions.create(model=groq_model, messages=messages)
    rich_print(f"[green]API response:[/green] [yellow]{response}[/yellow]")
    return response.choices[0].message.content

In [5]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Constructed messages for API: [{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called 
Akasha Air.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the 
answer, say so.\n"}, {'role': 'user', 'content': 'hi therre'}]

API response: ChatCompletion(id='chatcmpl-bfd4d8c5-fdf8-4505-b64b-ed98939d2a4e', 
choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can
I assist you today?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, 
tool_calls=None, reasoning='We need to give short courteous answer, no more than 1 sentence. The user says "hi 
therre". We should respond with a short courteous greeting, e.g., "Hello! How can I assist you today?" That\'s 1 
sentence. Good.'))], created=1781440002, model='openai/gpt-oss-20b', object='chat.completion', 
service_tier='on_demand', system_fingerprint='fp_3d587a02fb', usage=CompletionUsage(completion_tokens=71, 
prompt_tokens=117, total_tokens=188, 
completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, 
reasoning_tokens=53, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.041786391, 
prompt_time=0.005608888, completion_time=0.07501346, total_time=0.080622348), usage_breakdown=None, x_groq={'id': 
'req_01kv31jdw5efqver5r1bbn4ec7', 'seed': 806601194})

Constructed messages for API: [{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called 
Akasha Air.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the 
answer, say so.\n"}, {'role': 'user', 'content': 'hi therre'}, {'role': 'assistant', 'content': 'Hello! How can I 
assist you today?'}, {'role': 'user', 'content': 'how are you today'}]

API response: ChatCompletion(id='chatcmpl-c7d9dce7-db9d-4976-88d9-58366fbc91a4', 
choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="I'm just a 
virtual assistant, but I'm here and ready to help you!", refusal=None, role='assistant', annotations=None, 
audio=None, function_call=None, tool_calls=None, reasoning='Need short courteous answer. 1 sentence.'))], 
created=1781440010, model='openai/gpt-oss-20b', object='chat.completion', service_tier='on_demand', 
system_fingerprint='fp_80501ff3a1', usage=CompletionUsage(completion_tokens=34, prompt_tokens=140, 
total_tokens=174, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, 
audio_tokens=None, reasoning_tokens=10, rejected_prediction_tokens=None), prompt_tokens_details=None, 
queue_time=0.04351282, prompt_time=0.00678379, completion_time=0.034369573, total_time=0.041153363), 
usage_breakdown=None, x_groq={'id': 'req_01kv31jp56e7a908zpgkpms4z6', 'seed': 1549671072})

#### TOOLS
Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

In [6]:
# Let's start making an useful function

ticket_prices = {
    "bengaluru": "₹5000",
    "delhi": "₹4000",
    "mumbai": "₹4500",
    "varanasi": "₹3500",
    "kolkata": "₹3000",
    "chennai": "₹4000",
    "hyderabad": "₹4500"
}

def get_ticket_price(destination):
    price = ticket_prices.get(destination.lower())
    if price:
        return f"The ticket price to {destination} is {price}."
    else:
        return f"Sorry, we do not have flights to {destination} at the moment."

In [7]:
get_ticket_price("Bengaluru")

'The ticket price to Bengaluru is ₹5000.'

In [28]:
# There's a particular dictionary structure that's required to describe our function.

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "The city that the customer wants to travel to."
            }
        },
        "required": ["destination"],
        "additionalProperties": False
    }
}


In [29]:
# And this is included in a list of tools:

tools = [{
    "type": "function",
    "function": price_function
}]

#### LETTING GROQ TO USE OUR TOOL

There's some fiddly stuff to allow GROQ "to call our tool"

What we actually do is, give the LLM the opportunity to inform us, that it wants us to run the tool.

Here's how the new chat function looks:

In [31]:
# We've to write the handle_tool_calls function to handle the tool calls from the model. The structure of the message when the finish_reason is "tool_calls" is as follows:

def handle_tool_calls(message):
    responses = []
    tool_calls = message.tool_calls[0]
    rich_print(f"[green]Tool call received:[/green] [yellow]{tool_calls}[/yellow]")
    if tool_calls.function.name == "get_ticket_price":
        arguments = json.loads(tool_calls.function.arguments)
        city = arguments.get('destination')
        price_details = get_ticket_price(city)
        responses = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_calls.id
        }
    return responses


In [43]:
rich_print(tools)

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=groq_model, messages=messages, tools=tools)

    rich_print(response)

    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response = handle_tool_calls(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=groq_model, messages=messages)
        rich_print(f"[green]Final API response:[/green] [yellow]{response}[/yellow]")
        return response.choices[0].message.content
    elif response.choices[0].finish_reason == "stop":
        rich_print(f"[green]API response is complete without tool calls.[/green]")


[
    {
        'type': 'function',
        'function': {
            'name': 'get_ticket_price',
            'description': 'Get the price of a return ticket to the destination city.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'destination': {
                        'type': 'string',
                        'description': 'The city that the customer wants to travel to.'
                    }
                },
                'required': ['destination'],
                'additionalProperties': False
            }
        }
    }
]

In [44]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


ChatCompletion(
    id='chatcmpl-c30eec57-5c43-44c6-b5d6-7b1061bf52b2',
    choices=[
        Choice(
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ChatCompletionMessage(
                content='Hello! How can I help you with Akasha Air today?',
                refusal=None,
                role='assistant',
                annotations=None,
                audio=None,
                function_call=None,
                tool_calls=None,
                reasoning='The user says hi. The instructions: "Give short, courteous answers, no more than 1 
sentence. Always be accurate. If you don\'t know the answer, say so." We are to respond courteously. Just say 
"Hello! How can I assist you with Akasha Air today?" That\'s within 1 sentence. That\'s good. No function call 
needed.'
            )
        )
    ],
    created=1781441496,
    model='openai/gpt-oss-20b',
    object='chat.completion',
    service_tier='on_demand',
    system_fingerprint='fp_80501ff3a1',
    usage=CompletionUsage(
        completion_tokens=96,
        prompt_tokens=186,
        total_tokens=282,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=None,
            audio_tokens=None,
            reasoning_tokens=74,
            rejected_prediction_tokens=None
        ),
        prompt_tokens_details=None,
        queue_time=0.042360044,
        prompt_time=0.009015036,
        completion_time=0.098220254,
        total_time=0.10723529
    ),
    usage_breakdown=None,
    x_groq={'id': 'req_01kv3301h5fkfsv67n0mzjnz6p', 'seed': 887638484}
)

API response is complete without tool calls.

Traceback (most recent call last):
  File "d:\ujjwal\engineering_llm\.venv\Lib\site-packages\gradio\queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\ujjwal\engineering_llm\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\ujjwal\engineering_llm\.venv\Lib\site-packages\gradio\blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\ujjwal\engineering_llm\.venv\Lib\site-packages\gradio\blocks.py", line 1696, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\ujjwal\engineering_llm\.venv\Lib\site-packages\gradio\utils.py", line 882, in async_wrapper
    response = await f(*args, **kwargs)
               ^^^^^^^^^^^^^^

#### USING SQL DATABASE AND FRAMING A TOOL

In [45]:
import sqlite3

In [46]:
with sqlite3.connect("prices.db") as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [47]:
def get_ticket_price(city):
    rich_print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect("prices.db") as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [48]:
get_ticket_price("Bengaluru")

DATABASE TOOL CALLED: Getting price for Bengaluru

'No price data available for this city'

In [50]:
def set_ticket_price(city, price):
    with sqlite3.connect("prices.db") as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [51]:
ticket_prices = {
    "bengaluru": "₹5000",
    "delhi": "₹4000",
    "mumbai": "₹4500",
    "varanasi": "₹3500",
    "kolkata": "₹3000",
    "chennai": "₹4000",
    "hyderabad": "₹4500"
}

for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [52]:
get_ticket_price("varanasi")

DATABASE TOOL CALLED: Getting price for varanasi

'Ticket price to varanasi is $₹3500'

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


ChatCompletion(
    id='chatcmpl-291a7473-9dd0-4cf2-a9fc-7af696c16317',
    choices=[
        Choice(
            finish_reason='tool_calls',
            index=0,
            logprobs=None,
            message=ChatCompletionMessage(
                content=None,
                refusal=None,
                role='assistant',
                annotations=None,
                audio=None,
                function_call=None,
                tool_calls=[
                    ChatCompletionMessageFunctionToolCall(
                        id='fc_4afbd1d2-fe91-4928-9b01-aa8d6efe33c9',
                        function=Function(arguments='{"destination":"Varanasi"}', name='get_ticket_price'),
                        type='function'
                    )
                ],
                reasoning='We need to call the function get_ticket_price with destination "Varanasi". Provide 
result.'
            )
        )
    ],
    created=1781442537,
    model='openai/gpt-oss-20b',
    object='chat.completion',
    service_tier='on_demand',
    system_fingerprint='fp_80501ff3a1',
    usage=CompletionUsage(
        completion_tokens=46,
        prompt_tokens=191,
        total_tokens=237,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=None,
            audio_tokens=None,
            reasoning_tokens=20,
            rejected_prediction_tokens=None
        ),
        prompt_tokens_details=None,
        queue_time=0.045967752,
        prompt_time=0.010360758,
        completion_time=0.061033978,
        total_time=0.071394736
    ),
    usage_breakdown=None,
    x_groq={'id': 'req_01kv33zt6zf2xr0mtx2jdk762v', 'seed': 1628071704}
)

Tool call received: ChatCompletionMessageFunctionToolCall(id='fc_4afbd1d2-fe91-4928-9b01-aa8d6efe33c9', 
function=Function(arguments='{"destination":"Varanasi"}', name='get_ticket_price'), type='function')

DATABASE TOOL CALLED: Getting price for Varanasi

Final API response: ChatCompletion(id='chatcmpl-d06ae00d-c449-494b-8adf-a37575042834', 
choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The ticket 
price to Varanasi is ₹3500.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, 
tool_calls=None))], created=1781442538, model='openai/gpt-oss-20b', object='chat.completion', 
service_tier='on_demand', system_fingerprint='fp_3d587a02fb', usage=CompletionUsage(completion_tokens=16, 
prompt_tokens=184, total_tokens=200, completion_tokens_details=None, prompt_tokens_details=None, 
queue_time=0.044696452, prompt_time=0.009013238, completion_time=0.016388392, total_time=0.02540163), 
usage_breakdown=None, x_groq={'id': 'req_01kv33zteaf38rttjfsn7zv5k9', 'seed': 258814348})

ChatCompletion(
    id='chatcmpl-20880e3f-ab0e-4960-922a-530ea6fac516',
    choices=[
        Choice(
            finish_reason='tool_calls',
            index=0,
            logprobs=None,
            message=ChatCompletionMessage(
                content=None,
                refusal=None,
                role='assistant',
                annotations=None,
                audio=None,
                function_call=None,
                tool_calls=[
                    ChatCompletionMessageFunctionToolCall(
                        id='fc_20b149db-84ca-4963-8310-8821ec9e10dc',
                        function=Function(arguments='{"destination":"Mumbai"}', name='get_ticket_price'),
                        type='function'
                    )
                ],
                reasoning='The user asks: "how much cost for akasa air flight to mumbai from delhi". We need to 
provide ticket price. Use the function get_ticket_price with destination "Mumbai" probably. No extra properties.'
            )
        )
    ],
    created=1781442568,
    model='openai/gpt-oss-20b',
    object='chat.completion',
    service_tier='on_demand',
    system_fingerprint='fp_3d587a02fb',
    usage=CompletionUsage(
        completion_tokens=69,
        prompt_tokens=227,
        total_tokens=296,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=None,
            audio_tokens=None,
            reasoning_tokens=45,
            rejected_prediction_tokens=None
        ),
        prompt_tokens_details=None,
        queue_time=0.044725542,
        prompt_time=0.010999477,
        completion_time=0.070908944,
        total_time=0.081908421
    ),
    usage_breakdown=None,
    x_groq={'id': 'req_01kv340qw4f76s37khh024er41', 'seed': 23124266}
)

Tool call received: ChatCompletionMessageFunctionToolCall(id='fc_20b149db-84ca-4963-8310-8821ec9e10dc', 
function=Function(arguments='{"destination":"Mumbai"}', name='get_ticket_price'), type='function')

DATABASE TOOL CALLED: Getting price for Mumbai

Final API response: ChatCompletion(id='chatcmpl-a754e7a0-ae83-479b-845d-7b3870817ab2', 
choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='₹4500.', 
refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], 
created=1781442568, model='openai/gpt-oss-20b', object='chat.completion', service_tier='on_demand', 
system_fingerprint='fp_3d587a02fb', usage=CompletionUsage(completion_tokens=8, prompt_tokens=241, total_tokens=249,
completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.044909458, prompt_time=0.01174727, 
completion_time=0.008174926, total_time=0.019922196), usage_breakdown=None, x_groq={'id': 
'req_01kv340r4vf79v6hcad85w8vtj', 'seed': 180160500})